
# Applied Machine Learning, Clustering 2
## Pair trading: clustering stocks to find pairs that move together

**Course:** *Applied Machine Learning* · **Prepared by:** David Díaz, with assistance from Claude (Anthropic) · **Environment:** Google Colab

### Where we are coming from

In the previous notebook we clustered a bank's customers: rows that describe people, and the result was a map of the
customer base. Today the rows are **stocks** (the 500 in the S&P 500) and the clustering is not the end of the
analysis but an **intermediate step** inside an investment strategy. This is the most common use of clustering in
finance: not to describe, but to reduce an enormous problem to a manageable one.

The strategy is *pair trading*, one of the oldest in quantitative trading (the Morgan Stanley teams were using it in
the 1980s), and it also serves to learn three things that go beyond it: how to describe a stock with numbers so that
"similar" means what we want, why searching among thousands of candidates produces false discoveries even when nothing
is done wrong, and how a strategy is evaluated **out of sample** without fooling yourself.

### What you will learn today

1. What a cointegrated pair is, what the spread is, and why one can bet that it returns to its mean.
2. Why testing 125,000 pairs blindly produces thousands of "significant" pairs that are useless, and how clustering narrows the search.
3. Two ways of describing a stock: by its return and volatility, or by **how it moves together with the others** (principal components of the returns). And why the second is the one that works for this.
4. The Engle and Granger cointegration test, what its p-value says and what it does not say.
5. An honest backtest: pairs chosen with the 2021 to 2023 data and traded in 2024 and 2025, with fixed parameters and with rolling parameters.


In [ ]:

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import warnings, time

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from statsmodels.tsa.stattools import coint
import statsmodels.api as sm

warnings.filterwarnings("ignore")
pd.set_option("display.width", 130)
SEMILLA = 0



## 1. The problem: two stocks that walk together

Coca-Cola and Pepsi sell almost the same thing to almost the same people. Their prices are not equal, but the
**ratio** between them moves within a band: when one rises much more than the other, it is usually for something
transitory (a news item, a large flow), and the gap corrects within weeks. A *pair trade* bets on that correction: when
stock A is expensive relative to B, you **short A and buy B**, and you close the position when the gap returns to its
normal level. If the whole market goes up or down, the two legs offset each other: the bet is not on the market but on
the **relationship** between the two stocks. That is why it is called a **market-neutral** strategy.

For this to work three things are needed, and each has a technical name:

- That the relationship between the two prices is **stable over time**: that there is a number $\beta$ such that
  $\log P_A - \beta \log P_B$ oscillates around a fixed level instead of wandering off anywhere. We call that
  combination the **spread**, and the property, **cointegration**.
- That the spread **returns to its mean** reasonably fast (days or weeks, not years). That is measured with the
  **half-life** of the reversion.
- That the deviations are large enough to pay the trading costs.

The risk also has a name: that the relationship breaks. If one of the two companies changes its business, is acquired
or goes bankrupt, the spread never comes back and the short position loses without limit. That is why every pairs
strategy carries a **stop**: if the spread moves too far away, the position is closed and the loss is taken.

### Why machine learning is needed

With 500 stocks there are $500 \cdot 499 / 2 = 124,750$ possible pairs. Testing all of them for cointegration has two
problems. The first is time (each test costs a fraction of a second; that adds up to hours). The second is more serious,
and we saw it in the course when discussing **multiple testing**: if the test is wrong 5% of the time when there is no
cointegration, over 124,750 pairs it will declare about **6,200 pairs cointegrated that are not**, and there is no way
to tell them apart from the true ones by looking only at the p-value. It is the same phenomenon as testing a thousand
strategies and keeping the one that "worked".

Clustering attacks both things: it groups the stocks that are alike and searches for pairs **only within each group**.
There are fewer tests, and the ones that are run are between stocks that have an economic reason to move together.
What we have to decide is what "alike" means, and that decision is the heart of the notebook.

### The data and the golden rule

Adjusted closing prices of the S&P 500 stocks, daily, from 2021 to 2025. And the rule that is not negotiable, the same
one as in the dollar notebook: pairs are **discovered** with the 2021 to 2023 data and **traded** with the 2024 and
2025 data. If we search for pairs with all the data and then "test" the strategy on those same data, the result is as
fake as the Sharpe ratio of 8 in notebook 2.


In [ ]:

URL_DATOS = ""        # <-- link to the course file (sp500_precios.csv) if you have it; otherwise the course copy is tried and then Yahoo Finance
URL_CURSO = "https://raw.githubusercontent.com/daviddiazsolis/clustering_finance_playground/main/notebooks/data/sp500_precios.csv"
URL_SECTORES = "https://raw.githubusercontent.com/daviddiazsolis/clustering_finance_playground/main/notebooks/data/sp500_constituyentes.csv"

def normalizar_link(url):
    if "dropbox.com" in url:
        return url.replace("dl=0", "dl=1").replace("www.dropbox.com", "dl.dropboxusercontent.com")
    if "drive.google.com" in url and "/d/" in url:
        return "https://drive.google.com/uc?export=download&id=" + url.split("/d/")[1].split("/")[0]
    return url

precios, sectores = None, None
for url in [normalizar_link(URL_DATOS)] if URL_DATOS else [URL_CURSO]:
    try:
        precios = pd.read_csv(url, index_col=0, parse_dates=True)
        sectores = pd.read_csv(URL_SECTORES).set_index("ticker")["sector"]
        print("Prices read from", url[:60] + "...")
    except Exception as e:
        print("Could not read the course copy:", str(e)[:80])
if precios is None:
    import yfinance as yf, io, urllib.request
    html = urllib.request.urlopen(urllib.request.Request("https://en.wikipedia.org/wiki/List_of_S%26P_500_companies", headers={"User-Agent": "Mozilla/5.0"})).read().decode()
    sp = pd.read_html(io.StringIO(html))[0]
    sp["Symbol"] = sp["Symbol"].str.replace(".", "-", regex=False)
    sectores = sp.set_index("Symbol")["GICS Sector"]
    precios = yf.download(sp["Symbol"].tolist(), start="2021-01-01", end="2025-12-31", auto_adjust=True, progress=False)["Close"]
    print("Prices downloaded from Yahoo Finance")

print("Days:", len(precios), "  Stocks:", precios.shape[1], "  From", precios.index.min().date(), "to", precios.index.max().date())



## 2. Preparing the prices

Some stocks joined the index recently and do not have a full history; we drop them (more than 5% missing data).
Isolated gaps are filled with the last known price. Then we cut in time: **search** up to December 2023, **test**
from January 2024.


In [ ]:

precios = precios.loc[:, precios.isna().mean() < 0.05].ffill().dropna()
sectores = sectores.reindex(precios.columns)
busqueda = precios.loc[:"2023-12-31"]
prueba = precios.loc["2024-01-01":]
retornos = busqueda.pct_change().dropna()
print("Stocks with full history:", precios.shape[1])
print("Search:", len(busqueda), "days  |  Test:", len(prueba), "days")
print()
print("Stocks per sector:")
print(sectores.value_counts().to_string())



## 3. First attempt: describing each stock by its return and its volatility

The simplest way to describe a stock with numbers is with its average annual return and its annual volatility over the
search period. Two columns, 500 rows, and k-means on that. It is what most pair trading tutorials do, and it is worth
doing first to see with your own eyes what is wrong with it.


In [ ]:

rasgos1 = pd.DataFrame({"retorno": retornos.mean() * 252, "volatilidad": retornos.std() * np.sqrt(252)})
Z1 = StandardScaler().fit_transform(rasgos1)

filas = []
for k in [5, 10, 15, 20, 30, 40]:
    km = KMeans(n_clusters=k, n_init=10, random_state=SEMILLA).fit(Z1)
    filas.append(dict(K=k, silueta=silhouette_score(Z1, km.labels_), grupo_mas_grande=np.bincount(km.labels_).max()))
print(pd.DataFrame(filas).set_index("K").round(3).to_string())

K1 = 20
km1 = KMeans(n_clusters=K1, n_init=10, random_state=SEMILLA).fit(Z1)
rasgos1["grupo"] = km1.labels_
rasgos1["sector"] = sectores
fig = px.scatter(rasgos1.reset_index(names="ticker"), x="volatilidad", y="retorno", color=rasgos1["grupo"].astype(str), hover_data=["ticker", "sector"],
                 title=f"Stocks by return and volatility (2021 to 2023), {K1} k-means groups")
fig.update_layout(height=480, width=820, showlegend=False)
fig.show()



The groups are bands in the return-volatility plane. Nothing in them says that two stocks **move together**: a bank
and a pharmaceutical company with the same volatility and the same average return end up in the same group even if
they rise and fall on different days. We will search for pairs anyway, to measure the damage.

### The cointegration test

For each pair within a group, the **Engle and Granger** test does two things: it estimates by least squares the
relationship $\log P_A = \alpha + \beta \log P_B + \varepsilon$, and then it asks whether the residual
$\varepsilon$ (the spread) is **stationary**, that is, whether it oscillates around a fixed level instead of drifting
away without limit the way a price does. If the residual is stationary, the two prices are cointegrated. The `coint`
function in statsmodels returns the statistic and a p-value: with p < 0.05 we declare the pair cointegrated.

What the p-value does **not** say: how many times we are wrong when we repeat the test thousands of times. That count
we do ourselves.


In [ ]:

log_busqueda = np.log(busqueda)

def pares_cointegrados(grupos, tam_max=60, correlacion_min=None, corr=None):
    filas, pruebas = [], 0
    for g, miembros in grupos.groupby(grupos).groups.items():
        tickers = list(miembros)
        if len(tickers) < 2 or len(tickers) > tam_max:
            continue
        for i in range(len(tickers)):
            for j in range(i + 1, len(tickers)):
                a, b = tickers[i], tickers[j]
                if correlacion_min is not None and corr.loc[a, b] < correlacion_min:
                    continue
                pruebas += 1
                p = coint(log_busqueda[a], log_busqueda[b])[1]
                if p < 0.05:
                    filas.append(dict(grupo=g, a=a, b=b, valor_p=p))
    return pd.DataFrame(filas).sort_values("valor_p").reset_index(drop=True), pruebas

t0 = time.time()
pares1, n_pruebas1 = pares_cointegrados(rasgos1["grupo"])
print(f"Tests run: {n_pruebas1:,}   pairs with p < 0.05: {len(pares1)}   ({len(pares1) / n_pruebas1:.1%} of the tests; by pure chance we would expect 5%)   [{time.time() - t0:.0f} s]")
print()
pares1["sector_a"] = pares1["a"].map(sectores); pares1["sector_b"] = pares1["b"].map(sectores)
print("The ten pairs with the lowest p-value:")
print(pares1.head(10).round(5).to_string(index=False))



With groups built from return and volatility, 8% of the tests come out "cointegrated" against the 5% that would come
out by pure chance: most of the pairs found are false positives, and the ones with the lowest p-values are couples with
no economic relationship at all (an electronics retailer with an asset manager, a gas pipeline with a weapons
manufacturer). We keep this list to compare it in the backtest. Now let us describe the stocks better.



## 4. Second attempt: describing each stock by how it moves with the others

What matters for a pair is **co-movement**: that on the days one goes up, the other goes up too. That information is in
the correlation matrix of the returns, which for 500 stocks has 124,750 entries. It has to be summarized.

The standard summary is **principal component analysis (PCA)** of the daily returns. The idea, in one line: find the
few combinations of stocks that explain most of the joint movement. The first component is always "the market" (all
stocks with similar weights and the same sign); the next ones are sector contrasts (energy against technology,
utilities against consumer discretionary, and so on). Each stock is then described by its **loadings** on the first
components: ten numbers that say how much it responds to each of those common movements. Two stocks with similar
loadings rise and fall on the same days. That is exactly what we want to cluster. The PCA Playground in the hub has
the full geometry; here we use it as a tool.


In [ ]:

Rz = StandardScaler().fit_transform(retornos)              # each stock with returns of mean 0 and standard deviation 1
pca = PCA(n_components=10).fit(Rz)
print("Variance explained by each component:", (pca.explained_variance_ratio_ * 100).round(1), " (sum", round(pca.explained_variance_ratio_.sum() * 100, 1), "%)")
cargas = pd.DataFrame(pca.components_.T, index=retornos.columns, columns=[f"CP{i + 1}" for i in range(10)])
print()
print("Loadings of the first component: all with the same sign (it is the market). Sign check:", (cargas["CP1"] > 0).sum(), "positive loadings out of", len(cargas))
print("Loadings of the second component, extremes (it is a contrast between sectors):")
ext = pd.concat([cargas["CP2"].nsmallest(5), cargas["CP2"].nlargest(5)]).to_frame()
ext["sector"] = sectores.reindex(ext.index)
print(ext.round(3).to_string())


In [ ]:

Z2 = StandardScaler().fit_transform(cargas)
filas = []
for k in [5, 8, 10, 12, 15, 20, 30]:
    km = KMeans(n_clusters=k, n_init=10, random_state=SEMILLA).fit(Z2)
    filas.append(dict(K=k, silueta=silhouette_score(Z2, km.labels_), grupo_mas_grande=np.bincount(km.labels_).max()))
print(pd.DataFrame(filas).set_index("K").round(3).to_string())

K2 = 12
km2 = KMeans(n_clusters=K2, n_init=10, random_state=SEMILLA).fit(Z2)
grupos2 = pd.Series(km2.labels_, index=retornos.columns, name="grupo")
tabla = pd.crosstab(grupos2, sectores.reindex(grupos2.index))
print()
print("Stocks of each sector in each group (nobody told the algorithm which sector each one was):")
print(tabla.to_string())



The table is the central result of the notebook: clustering by co-movement, the groups **recover the sectors**. There
is a group that is almost pure utilities, another almost pure real estate, another of banks, another of semiconductors.
The algorithm never saw the word "sector"; it found it in the returns. (If the groups coincide with the GICS
classification, what is the clustering for? Because GICS classifies by what the company **declares** it does and the
clustering by how it **behaves**; when they differ, the one that matters for a pair is the second.)

### Pairs within the groups, with economic filters

We search for pairs within these groups, and this time we add three filters that a trader would demand before looking
at p-values:

- correlation of daily returns of at least 0.6 (that they really move together);
- $\beta$ between 0.5 and 2 (that one leg is not ten times the other);
- half-life of the spread under 60 trading days. The half-life is estimated with a regression of the change in the
  spread on its lagged level, $\Delta s_t = \gamma\, s_{t-1} + u_t$: if $\gamma < 0$ the spread reverts, and
  $\text{half-life} = -\ln 2 / \gamma$ is how many days it takes to cover half of the way back.


In [ ]:

correlacion = retornos.corr()

def beta_y_vida_media(a, b):
    ya, yb = log_busqueda[a], log_busqueda[b]
    beta = sm.OLS(ya, sm.add_constant(yb)).fit().params[b]
    s = ya - beta * yb
    ds, rez = s.diff().dropna(), s.shift().dropna()
    gamma = sm.OLS(ds, sm.add_constant(rez.loc[ds.index])).fit().params.iloc[1]
    return beta, (-np.log(2) / gamma if gamma < 0 else np.inf)

t0 = time.time()
pares2, n_pruebas2 = pares_cointegrados(grupos2, tam_max=120, correlacion_min=0.6, corr=correlacion)
print(f"Tests run: {n_pruebas2:,}   pairs with p < 0.05: {len(pares2)}   ({len(pares2) / n_pruebas2:.1%})   [{time.time() - t0:.0f} s]")
pares2[["beta", "vida_media"]] = [beta_y_vida_media(a, b) for a, b in zip(pares2["a"], pares2["b"])]
pares2["correlacion"] = [correlacion.loc[a, b] for a, b in zip(pares2["a"], pares2["b"])]
pares2["sector_a"] = pares2["a"].map(sectores); pares2["sector_b"] = pares2["b"].map(sectores)
candidatos = pares2[(pares2["beta"].between(0.5, 2)) & (pares2["vida_media"] < 60)].sort_values("vida_media").reset_index(drop=True)
print("Pairs that pass the three filters:", len(candidatos))
print()
print("The ten with the fastest reversion:")
print(candidatos.head(10)[["a", "b", "sector_a", "sector_b", "valor_p", "correlacion", "beta", "vida_media"]].round(3).to_string(index=False))



Now the couples make sense: two Michigan utilities (CMS and DTE), two homebuilders (DHI and LEN), two oil companies
(COP and EOG), two banks (BAC and PNC), two makers of semiconductor equipment. This is what the clustering was
supposed to deliver: candidates with an economic reason to walk together.

## 5. The backtest: trading the pairs in 2024 and 2025

The trading rule is the classic one. The spread is computed with the $\beta$ estimated in the search period, it is
standardized as a **z-score** (how many standard deviations it is from its mean) and:

- if $z$ crosses above 2, the spread is high: sell A and buy $\beta$ of B;
- if it crosses below -2, it is low: buy A and sell $\beta$ of B;
- the position is closed when $|z| < 0.5$ (it returned to the mean) or when $|z| > 4$ (stop: the relationship broke),
  and it is not reopened until $z$ crosses the threshold again from the inside.

The daily return of the position is that of the spread, divided by the capital committed in the two legs
($1 + \beta$) so that it is comparable across pairs. There is one more decision: **the mean and standard deviation of
the spread, are they fixed with the search data or recomputed with a rolling window?** We try both.

And a precaution that defines the notebook: we are not going to look only at the 20 "best" pairs of each method,
because "best" was decided with search data and the ranking may be pure noise. We will also look at **all** the pairs
each method found and, as a control, **pairs chosen at random** among the 489 stocks, traded with the same rule. If
the random pairs perform as well as ours, what we are measuring is the rule and the period, not the selection.


In [ ]:

log_precios = np.log(precios)

def operar_par(a, b, beta, ventana_movil=None, entrada=2.0, salida=0.5, stop=4.0):
    spread = log_precios[a] - beta * log_precios[b]
    if ventana_movil is None:                                   # fixed parameters, estimated in the search period
        s_busq = spread.loc[busqueda.index]
        z = (spread - s_busq.mean()) / s_busq.std()
    else:                                                       # rolling parameters: last `ventana_movil` days
        z = (spread - spread.rolling(ventana_movil).mean()) / spread.rolling(ventana_movil).std()
    z = z.loc[prueba.index]
    posicion, actual, previo = pd.Series(0.0, index=z.index), 0, 0.0
    for dia in z.index:
        v = z[dia]
        if actual == 0 and v > entrada and previo <= entrada: actual = -1        # crosses upward: short the spread
        elif actual == 0 and v < -entrada and previo >= -entrada: actual = 1    # crosses downward: long the spread
        elif actual != 0 and (abs(v) < salida or abs(v) > stop): actual = 0     # it came back, or it broke
        posicion[dia] = actual
        previo = v
    r_a, r_b = prueba[a].pct_change(), prueba[b].pct_change()
    retorno = (posicion.shift() * (r_a - beta * r_b) / (1 + beta)).fillna(0)
    return retorno, z, posicion

def resumen_cartera(lista, ventana_movil, costo=0.0):
    rets = {}
    for p in lista.itertuples():
        ret, z, pos = operar_par(p.a, p.b, p.beta, ventana_movil)
        rets[f"{p.a}-{p.b}"] = ret - costo * (pos.diff().abs() > 0).astype(float)
    rets = pd.DataFrame(rets)
    total = (1 + rets).prod() - 1
    cartera = rets.mean(axis=1)
    return dict(pares=len(rets.columns), pares_positivos=(total > 0).mean(), mediana_por_par=total.median(), retorno_cartera=(1 + cartera).prod() - 1,
                volatilidad_cartera=cartera.std() * np.sqrt(252), sharpe_cartera=cartera.mean() / cartera.std() * np.sqrt(252)), rets

# the lists to compare
pares1["beta"] = [beta_y_vida_media(a, b)[0] for a, b in zip(pares1["a"], pares1["b"])]
todos1 = pares1[pares1["beta"].between(0.5, 2)]
rng = np.random.default_rng(SEMILLA)
acciones = list(precios.columns)
azar = pd.DataFrame([(acciones[i], acciones[j]) for i, j in rng.integers(0, len(acciones), (400, 2)) if i != j][:250], columns=["a", "b"])
azar["beta"] = [beta_y_vida_media(a, b)[0] for a, b in zip(azar["a"], azar["b"])]
azar = azar[azar["beta"].between(0.5, 2)]

LISTAS = {"return and volatility: 20 with the lowest p-value": todos1.head(20), "return and volatility: all pairs": todos1,
          "co-movement: 20 with the fastest reversion": candidatos.head(20), "co-movement: all pairs": candidatos,
          "random pairs (control)": azar}
filas = {}
for nombre, lista in LISTAS.items():
    for vent, etiqueta in [(None, "fixed parameters"), (60, "60-day rolling window")]:
        filas[(nombre, etiqueta)], _ = resumen_cartera(lista, vent)
comparacion = pd.DataFrame(filas).T
print("Out-of-sample performance (2024 and 2025), without transaction costs:")
print(comparacion.round(3).to_string())



The table has to be read from the bottom up.

**The random pairs end up at zero**: they lose with fixed parameters and gain nothing with a rolling window (half of
the pairs end positive, the other half negative). That is the reference point: this is how the rule performs when the
selection contributes nothing.

**All the pairs of each method perform almost like random.** The return-and-volatility pairs and the co-movement
pairs stay close to zero with a rolling window and in negative territory with fixed parameters. If the selection had
found pairs that truly revert, the full portfolio should separate from the control, and it does not do so clearly.

**The 20 "best" of each method look better, and that is the dangerous part.** They are 20 out of hundreds, chosen by a
ranking that was computed with the search data; their results have much more dispersion (look at the portfolio
volatility) and most of the difference from the total is selection noise. An analyst who reports only that row is
doing, without meaning to, the same thing as the one who tests a thousand strategies and shows one.

**Fixed parameters are worse than rolling ones in every row.** The mean and standard deviation of the spread from 2021
to 2023 do not describe 2024 and 2025: the spread drifts, the z-score stays out of range for months and the rule opens
at a loss. It is the problem of the whole course with another face: estimated parameters age.

The honest conclusion, then: in 2024 and 2025, with this simple rule and no costs, pair trading on the S&P 500 **does
not make money**, not even with good selection. It is not a failure of the clustering, which did its job (candidates
with economic sense and a search fifty times smaller), but of the premise: these were two years of strong trends
within sectors (semiconductors, banks, utilities), the worst climate for a reversion strategy. A backtest that says
"no" clearly is worth more than one that says "yes" because it looked at the future.


In [ ]:

# One pair up close: the spread, the rolling z-score and the positions
par = candidatos.iloc[0]
ret, z, pos = operar_par(par.a, par.b, par.beta, ventana_movil=60)
fig = go.Figure()
fig.add_scatter(x=z.index, y=z, name="rolling z-score", line=dict(color="#2563eb"))
for nivel, color in [(2, "#dc2626"), (-2, "#dc2626"), (0.5, "#9ca3af"), (-0.5, "#9ca3af"), (4, "#111827"), (-4, "#111827")]:
    fig.add_hline(y=nivel, line=dict(color=color, dash="dot", width=1))
fig.add_scatter(x=pos.index, y=pos * 3, name="position (x3 to make it visible)", line=dict(color="#f59e0b", shape="hv"))
fig.update_layout(title=f"{par.a} against {par.b} ({par.sector_a}): z-score of the spread in 2024 and 2025 and positions taken", height=420, width=860, yaxis_title="z")
fig.show()
print(f"Cumulative return of the pair in the test period: {(1 + ret).prod() - 1:.1%}   trades (openings and closings): {int((pos.diff().abs() > 0).sum())}")


In [ ]:

_, rets_movil = resumen_cartera(candidatos, 60)
acumulado = (1 + rets_movil).cumprod()
fig = go.Figure()
for c in acumulado.columns:
    fig.add_scatter(x=acumulado.index, y=acumulado[c], name=c, line=dict(width=1), opacity=0.35, showlegend=False)
fig.add_scatter(x=acumulado.index, y=(1 + rets_movil.mean(axis=1)).cumprod(), name="portfolio of all co-movement pairs", line=dict(color="black", width=3))
_, rets_azar = resumen_cartera(azar, 60)
fig.add_scatter(x=acumulado.index, y=(1 + rets_azar.mean(axis=1)).cumprod(), name="portfolio of random pairs", line=dict(color="#dc2626", width=3, dash="dash"))
fig.update_layout(title="Cumulative capital per pair (gray) and of the equal-weighted portfolios (2024 and 2025, rolling window, no costs)", height=480, width=900, yaxis_title="capital (1 = start)")
fig.show()



## 6. What to take away

1. Clustering in finance is usually an intermediate step: it reduces 125,000 candidates to a few hundred with economic sense and, along the way, limits the multiple testing problem.
2. "Similar" depends on how the stocks are described. Return and volatility say nothing about moving together; the loadings on the principal components of the returns do, and they recover the sectors without anyone naming them.
3. Cointegration is tested with Engle and Granger, but the p-value is not enough: with thousands of tests you also have to demand correlation, a reasonable $\beta$ and a short half-life.
4. Out of sample, parameters age: the spread drifts, and a z-score with fixed mean and standard deviation opens positions that never close. Rolling windows are the minimum correction.
5. An honest backtest is designed before looking at the test data, compares against a random control, and looks at all the pairs and not only the 20 best of a ranking that was made with the same data. In 2024 and 2025 the answer was "no", and that answer is information.



## Exercises

1. **Declared sectors against discovered sectors.** Repeat the pair search using directly the GICS sector of each
   stock as the groups (instead of the k-means groups), with the same filters. How many tests are run, how many pairs
   pass, and how does the portfolio of all those pairs perform with a rolling window? What does clustering add over the
   official classification?
2. **Transaction costs.** Each opening or closing of a position costs 10 basis points of the pair's capital (both
   legs). Subtract them from the daily return of each pair and recompute the row "co-movement: all pairs" of
   section 5. How much does the rule cost in trades per pair per year?
3. **The rolling window.** Repeat the backtest of all the co-movement pairs with windows of 20, 40, 60, 120 and 250
   days and plot the portfolio Sharpe ratio against the window. What happens with very short and very long windows, and
   why can you not pick the best one by looking at this table?



## Solutions


In [ ]:

# Exercise 1: groups = GICS sector
t0 = time.time()
pares_gics, n_gics = pares_cointegrados(sectores.reindex(retornos.columns), tam_max=120, correlacion_min=0.6, corr=correlacion)
pares_gics[["beta", "vida_media"]] = [beta_y_vida_media(a, b) for a, b in zip(pares_gics["a"], pares_gics["b"])]
cand_gics = pares_gics[(pares_gics["beta"].between(0.5, 2)) & (pares_gics["vida_media"] < 60)].sort_values("vida_media").reset_index(drop=True)
res_gics, _ = resumen_cartera(cand_gics, 60)
print(f"With GICS sectors: {n_gics:,} tests, {len(pares_gics)} cointegrated, {len(cand_gics)} pass the filters   [{time.time() - t0:.0f} s]")
print(pd.DataFrame({"k-means on co-movement": comparacion.loc[("co-movement: all pairs", "60-day rolling window")], "GICS sectors": pd.Series(res_gics)}).round(3).to_string())



The GICS sectors produce a search of the same size and a similar portfolio: when the official classification is good,
the clustering reproduces it and does not add much. Its value shows up where there is no classification (other markets,
new assets, bonds, cryptocurrencies) or where the classification lies (a company classified as industrial that behaves
like a technology company). The algorithm does not need someone to have classified beforehand.


In [ ]:

# Exercise 2: costs of 10 basis points per opening or closing
COSTO = 0.0010
sin, rets_sin = resumen_cartera(candidatos, 60)
con, _ = resumen_cartera(candidatos, 60, costo=COSTO)
ops = np.mean([(operar_par(p.a, p.b, p.beta, 60)[2].diff().abs() > 0).sum() for p in candidatos.itertuples()])
print(f"Average trades per pair over the two test years: {ops:.1f}  (about {ops / 2:.0f} per year)")
print(pd.DataFrame({"no costs": sin, "with 10 bp per trade": con}).loc[["pares_positivos", "retorno_cartera", "sharpe_cartera"]].round(3).to_string())



With a rolling window the rule trades quite a lot (a half-life of two weeks means frequent entries and exits), and ten
basis points per trade eat more than one point of return per year. Since the portfolio was already close to zero before
costs, after costs it is clearly negative. That is why the funds that live off this compete on execution costs and
speed, not on the selection algorithm.


In [ ]:

# Exercise 3: sensitivity to the rolling window
filas = []
for v in [20, 40, 60, 120, 250]:
    r_, _ = resumen_cartera(candidatos, v)
    filas.append(dict(ventana=v, sharpe=r_["sharpe_cartera"], retorno=r_["retorno_cartera"], pares_positivos=r_["pares_positivos"]))
sens = pd.DataFrame(filas).set_index("ventana")
print(sens.round(3).to_string())
fig = px.line(sens.reset_index(), x="ventana", y="sharpe", markers=True, title="Sharpe ratio of the all-pairs portfolio by rolling z-score window")
fig.update_layout(height=360, width=640)
fig.show()



With all the pairs, no window changes the conclusion: short ones (20 days) compute the z-score on very few data
points, any move looks like "two standard deviations" and the rule overtrades; very long ones (250 days) approach the
fixed-parameter case, the drift comes back, and it is the one that performs worst. And the reason why the window cannot
be chosen from this table is the usual one: the table is computed over 2024 and 2025, which is the test period. Picking
the parameter that performed best there and reporting that result would be looking at the future again; the window is
chosen with search data, and the test is looked at only once.
